1. Conexão com SQL Server

Nesta etapa será realizada a conexão entre o Databricks e o banco de dados SQL Server utilizando o protocolo JDBC.

Para manter as credenciais protegidas, as informações de acesso ao banco não serão inseridas diretamente no código. Elas estão armazenadas em um Databricks Secret Scope e serão recuperadas de forma segura durante a execução do notebook.

As credenciais utilizadas nesta conexão são o endereço do servidor, o nome do banco de dados, o usuário e a senha.

In [0]:
jdbc_hostname = dbutils.secrets.get(
    scope="mercadata-sqlserver",
    key="jdbc_hostname"
)

jdbc_database = dbutils.secrets.get(
    scope="mercadata-sqlserver",
    key="jdbc_database"
)

jdbc_username = dbutils.secrets.get(
    scope="mercadata-sqlserver",
    key="jdbc_username"
)

jdbc_password = dbutils.secrets.get(
    scope="mercadata-sqlserver",
    key="jdbc_password"
)

2. Validação do carregamento das credenciais

Nesta etapa será validado se as credenciais armazenadas no Secret Scope foram recuperadas corretamente pelo notebook.

Por segurança, os valores das credenciais não serão exibidos. Será verificado apenas se cada variável foi carregada e possui um valor disponível para utilização na conexão com o banco de dados.

In [0]:
print("JDBC Hostname carregado:", jdbc_hostname is not None)
print("JDBC Database carregado:", jdbc_database is not None)
print("JDBC Username carregado:", jdbc_username is not None)
print("JDBC Password carregado:", jdbc_password is not None)

#Isso é uma boa prática porque comprova que o notebook conseguiu acessar os secrets sem expor nenhuma informação sensível.

3. Configuração da conexão JDBC

Nesta etapa será configurada a conexão JDBC (Java Database Connectivity), utilizada para permitir que o Databricks se comunique com o banco de dados SQL Server.

A URL JDBC será construída a partir do endereço do servidor e do banco de dados recuperados do Secret Scope. Dessa forma, as informações sensíveis permanecem protegidas e não são inseridas diretamente no código.

In [0]:
jdbc_url = f"jdbc:sqlserver://{jdbc_hostname};databaseName={jdbc_database}"

#Por que começamos pela jdbc_url?
#A conexão JDBC precisa saber basicamente:
#Servidor → qual SQL Server acessar Banco → qual database utilizar

4. Configuração das credenciais JDBC

Nesta etapa serão definidas as propriedades de autenticação utilizadas pelo JDBC para acessar o banco de dados SQL Server.

O usuário e a senha são recuperados anteriormente por meio do Databricks Secret Scope e serão utilizados diretamente na conexão, sem serem exibidos no notebook.

In [0]:
connection_properties = {
    "user": jdbc_username,
    "password": jdbc_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

#O connection_properties é um dicionário que informa ao JDBC:
#-user → usuário do banco;
#-password → senha do banco;
#-driver → driver responsável pela comunicação com o SQL Server.

5. Teste de conexão com o SQL Server

Nesta etapa será realizado um teste de conexão com o banco de dados SQL Server utilizando as configurações JDBC definidas anteriormente.

O objetivo é verificar se o Databricks consegue estabelecer comunicação com o banco utilizando as credenciais armazenadas no Secret Scope.

In [0]:
teste_conexao = spark.read.jdbc(
    url=jdbc_url,
    table="(SELECT 1 AS conexao) AS teste",
    properties=connection_properties
)

display(teste_conexao)

#Ele envia para o SQL Server uma consulta equivalente a:
#SELECT 1 AS conexao

Agora temos uma confirmação real de que o Databricks conseguiu acessar o SQL Server usando as credenciais armazenadas no Secret Scope.

Até aqui, seu fluxo ficou bem organizado:

![image_1790555691188.png](./image_1790555691188.png "image_1790555691188.png")


6. Leitura da tabela ecommerce_pedidos

Com a conexão JDBC validada, será realizada a leitura da tabela ecommerce_pedidos, armazenada no banco de dados SQL Server.

Os dados serão carregados em um DataFrame Spark para que sua estrutura e seus registros possam ser analisados posteriormente no Databricks.

In [0]:
df_pedidos = spark.read.jdbc(
    url=jdbc_url,
    table="ecommerce_pedidos",
    properties=connection_properties
)

7. Verificação da estrutura dos dados

Nesta etapa será analisada a estrutura do DataFrame df_pedidos, permitindo identificar as colunas disponíveis e os respectivos tipos de dados.

Essa verificação é importante antes de iniciar a análise exploratória, pois permite compreender como os dados estão organizados e identificar possíveis necessidades de tratamento ou transformação.

In [0]:
df_pedidos.printSchema()

8. Visualização inicial dos dados

Após verificar a estrutura da tabela, será realizada uma visualização inicial dos registros da ecommerce_pedidos.

O objetivo é observar os dados na prática, identificar o formato dos valores e verificar se existem características que possam exigir tratamento antes da análise exploratória.

In [0]:
display(df_pedidos.limit(10))

9. Quantidade de registros

Nesta etapa será verificada a quantidade total de registros presentes na tabela ecommerce_pedidos.

Essa informação permite compreender o volume de dados disponível para a análise e serve como uma referência para as próximas etapas da exploração.

In [0]:
print("Quantidade de pedidos:", df_pedidos.count())

10. Verificação de valores nulos

Nesta etapa será realizada a identificação de valores nulos nas colunas da tabela ecommerce_pedidos.

A análise de valores ausentes é importante para avaliar a qualidade dos dados e identificar possíveis colunas que necessitem de tratamento antes da realização das análises.

In [0]:
from pyspark.sql.functions import col, sum

df_pedidos.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df_pedidos.columns
    ]
).show()